
<div style="background:#eff6ff; padding:22px 26px; border-radius:14px; border-left:6px solid #1d4ed8; margin:18px 0;">
<h1 style="color:#1d4ed8; margin:0 0 4px 0;">MaisonDeLUX — Analyse du marché immobilier (Casablanca)</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>De la donnée brute à l'analyse décisionnelle et à la modélisation prédictive</p>
</div>



##  Sommaire

1. **Contexte et objectifs**
2. **Importation et compréhension des données**
3. **Nettoyage et préparation des données**
4. **Analyse exploratoire (EDA)**
5. **Questions Data Analyst — réponses à partir des données**
6. **Préparation pour le Machine Learning**
7. **Comparaison et optimisation des modèles**
8. **Conclusion et recommandations**

> **Fil conducteur du rapport :** comprendre le marché immobilier à partir des annonces Mubawab, identifier les variables qui structurent les prix, répondre à des questions métier concrètes, puis préparer les données pour l'estimation prédictive.



<div style="background:#ecfdf5; padding:22px 26px; border-radius:14px; border-left:6px solid #0f766e; margin:18px 0;">
<h1 style="color:#0f766e; margin:0 0 4px 0;">1. Contexte et objectifs</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Comprendre le marché avant de prédire les prix</p>
</div>



### Contexte

Ce notebook s'inscrit dans le cadre de l'analyse et de l'estimation des prix de l'immobilier. Les données utilisées ont été collectées à partir du site web Mubawab : https://www.mubawab.ma/ via des techniques de web scraping. Elles représentent un échantillon de 1172 annonces immobilières, principalement concentrées sur la région de Casablanca (Anfa, Maârif, Bourgogne, etc.).

### Objectifs analytiques

- comprendre la structure du marché représenté par les annonces ;
- mesurer la distribution des prix et du **prix au m²** ;
- étudier le rôle de la superficie, du nombre de pièces et des autres caractéristiques ;
- comparer les quartiers et les types de biens ;
- transformer les informations textuelles en variables exploitables ;
- répondre à des **questions métier/Data Analyst** avant de passer au Machine Learning ;
- préparer un jeu de données cohérent pour l'estimation du prix.

### Variables principales

| Variable | Rôle |
|---|---|
| `Type` | Type de bien |
| `Localisation` | Quartier / zone |
| `Price` | Prix de vente — variable cible |
| `Area` | Superficie en m² |
| `Rooms` | Nombre de pièces |
| `Bedrooms` | Nombre de chambres |
| `Bathrooms` | Nombre de salles de bain |
| `Floor` | Étage |
| `Other_tags` | Informations sur l'état et l'ancienneté |
| `Price_m2` | Prix calculé par m² |



<div style="background:#f5f3ff; padding:22px 26px; border-radius:14px; border-left:6px solid #7c3aed; margin:18px 0;">
<h1 style="color:#7c3aed; margin:0 0 4px 0;">2. Importation des bibliothèques</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Préparer l'environnement d'analyse</p>
</div>


In [80]:
import math
import re

from statistics import linear_regression
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    MinMaxScaler,
    Normalizer,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_curve,
)

import joblib

In [81]:
df = pd.read_csv('mubawab_listings_clean.csv')
df.head()

,Type,Localisation,Price,Area,Rooms,Bedrooms,Bathrooms,Floor,Other_tags,Price_m2
0,Appartements,Casablanca Finance City,22000,219.0,4.0,3.0,3.0,11.0,"['Nouveau', ""Moins d'un an""]",100.456621
1,Appartements,Anfa,1230000,86.0,3.0,2.0,2.0,1.0,"['Bon état', '10-20 ans']",14302.325581
2,Appartements,La Gironde,900000,86.0,3.0,2.0,1.0,5.0,"['Bon état', '10-20 ans']",10465.116279
3,Appartements,Bourgogne Ouest,2400000,180.0,5.0,3.0,3.0,3.0,"['Bon état', '10-20 ans']",13333.333333
4,Appartements,Les princesses,1550000,120.0,5.0,3.0,2.0,5.0,"['Bon état', '5-10 ans']",12916.666667



<div style="background:#eff6ff; padding:22px 26px; border-radius:14px; border-left:6px solid #2563eb; margin:18px 0;">
<h1 style="color:#2563eb; margin:0 0 4px 0;">2. Compréhension des données</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Identifier la structure, les types de variables et la qualité initiale.</p>
</div>


In [82]:
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])
df.head(10)

Nombre de lignes : 1172
Nombre de colonnes : 10


,Type,Localisation,Price,Area,Rooms,Bedrooms,Bathrooms,Floor,Other_tags,Price_m2
0,Appartements,Casablanca Finance City,22000,219.0,4.0,3.0,3.0,11.0,"['Nouveau', ""Moins d'un an""]",100.456621
1,Appartements,Anfa,1230000,86.0,3.0,2.0,2.0,1.0,"['Bon état', '10-20 ans']",14302.325581
2,Appartements,La Gironde,900000,86.0,3.0,2.0,1.0,5.0,"['Bon état', '10-20 ans']",10465.116279
3,Appartements,Bourgogne Ouest,2400000,180.0,5.0,3.0,3.0,3.0,"['Bon état', '10-20 ans']",13333.333333
4,Appartements,Les princesses,1550000,120.0,5.0,3.0,2.0,5.0,"['Bon état', '5-10 ans']",12916.666667
5,Appartements,Bourgogne Ouest,1450000,113.0,4.0,3.0,2.0,1.0,"['Bon état', '10-20 ans']",12831.858407
6,Appartements,Triangle d'Or,4800000,245.0,4.0,3.0,3.0,3.0,"['Bon état', '10-20 ans']",19591.836735
7,Appartements,Anassi,330000,50.0,3.0,2.0,1.0,1.0,"['Bon état', '5-10 ans']",6600.000000
8,Villas,Californie,7500000,400.0,8.0,5.0,4.0,0.0,['Bon état'],18750.000000
9,Appartements,Maârif,1750000,113.0,3.0,2.0,2.0,4.0,"['Nouveau', '1-5 ans']",15486.725664


In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1172 entries, 0 to 1171
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Type          1172 non-null   str    
 1   Localisation  1172 non-null   str    
 2   Price         1172 non-null   int64  
 3   Area          1172 non-null   float64
 4   Rooms         1172 non-null   float64
 5   Bedrooms      1172 non-null   float64
 6   Bathrooms     1172 non-null   float64
 7   Floor         1172 non-null   float64
 8   Other_tags    1172 non-null   str    
 9   Price_m2      1172 non-null   float64
dtypes: float64(6), int64(1), str(3)
memory usage: 140.7 KB


In [84]:
categorical_cols = df.select_dtypes(include=["str"])
categorical_cols

,Type,Localisation,Other_tags
0,Appartements,Casablanca Finance City,"['Nouveau', ""Moins d'un an""]"
1,Appartements,Anfa,"['Bon état', '10-20 ans']"
2,Appartements,La Gironde,"['Bon état', '10-20 ans']"
3,Appartements,Bourgogne Ouest,"['Bon état', '10-20 ans']"
4,Appartements,Les princesses,"['Bon état', '5-10 ans']"
...,...,...,...
1167,Villas,Californie,['Bon état']
1168,Appartements,Val Fleury,['Bon état']
1169,Appartements,Hermitage,"['Nouveau', ""Moins d'un an""]"
1170,Appartements,Hay Arrahma,"['Bon état', '5-10 ans']"


In [85]:
numeric_cols = df.select_dtypes(include=['number']).columns
numeric_cols

Index(['Price', 'Area', 'Rooms', 'Bedrooms', 'Bathrooms', 'Floor', 'Price_m2'], dtype='str')

In [86]:
df['Type'].unique()

<ArrowStringArray>
['Appartements', 'Villas', 'Riad', 'Maisons']
Length: 4, dtype: str

In [87]:
df['Localisation'].unique()

<ArrowStringArray>
[    'Casablanca Finance City',                        'Anfa',
                  'La Gironde',             'Bourgogne Ouest',
              'Les princesses',               'Triangle d'Or',
                      'Anassi',                  'Californie',
                      'Maârif',                      'Racine',
                    'Gauthier',                  'Val Fleury',
                'Sidi Maarouf',                    'Ain Diab',
            'Racine Extension',                     'Palmier',
                       'Oasis',             'CIL (Hay Salam)',
                  'Les Crêtes',               'Bourgogne Est',
            'Maârif Extension',          'Ain Diab Extension',
                  'Hay Alamal',                 'Mers Sultan',
                  'Beauséjour',                    'Laymoune',
   'Longchamps (Hay Al Hanâa)',          'El Manar - El Hank',
                       'Oulfa',                'Les Hôpitaux',
                 'Derb Ghalef',     

In [88]:
df['Other_tags'].unique()

<ArrowStringArray>
[ '['Nouveau', "Moins d'un an"]',     '['Bon état', '10-20 ans']',
      '['Bon état', '5-10 ans']',                  '['Bon état']',
        '['Nouveau', '1-5 ans']',                   '['Nouveau']',
    '['À rénover', '30-50 ans']',       '['Nouveau', '5-10 ans']',
       '['Bon état', '1-5 ans']',    '['À rénover', '20-30 ans']',
     '['Bon état', '20-30 ans']',                            '[]',
    '['À rénover', '10-20 ans']', '['Bon état', "Moins d'un an"]',
                 '['À rénover']',      '['Nouveau', '10-20 ans']',
     '['À rénover', '5-10 ans']',     '['Bon état', '30-50 ans']',
      '['Nouveau', '20-30 ans']',      '['À rénover', '1-5 ans']',
    '['À rénover', '50-70 ans']']
Length: 21, dtype: str

In [89]:
numeric_cols = df.select_dtypes(include="number").columns

print("Variables numériques :", len(numeric_cols))

df[numeric_cols].describe().T

Variables numériques : 7


,count,mean,std,min,25%,50%,75%,max
Price,1172.0,4.954876e+06,7.401288e+06,1600.000000,1.221922e+06,2.300000e+06,5.990000e+06,7.140000e+07
Area,1172.0,3.155691e+02,4.659344e+02,32.000000,9.100000e+01,1.385000e+02,3.602500e+02,5.400000e+03
Rooms,1172.0,4.814846e+00,2.298929e+00,1.000000,3.000000e+00,4.000000e+00,6.000000e+00,1.600000e+01
Bedrooms,1172.0,3.029863e+00,1.370656e+00,1.000000,2.000000e+00,3.000000e+00,4.000000e+00,1.800000e+01
Bathrooms,1172.0,2.408703e+00,1.208153e+00,1.000000,2.000000e+00,2.000000e+00,3.000000e+00,1.000000e+01
Floor,1172.0,2.277304e+00,2.206828e+00,0.000000,0.000000e+00,2.000000e+00,4.000000e+00,2.000000e+01
Price_m2,1172.0,1.568007e+04,7.721872e+03,33.333333,1.172279e+04,1.508853e+04,1.833494e+04,1.108108e+05


In [90]:
categorical_cols = df.select_dtypes(
    include=["object", "category", "bool"]
).columns

categorical_summary = pd.DataFrame({
    "dtype": df[categorical_cols].dtypes,
    "non_null": df[categorical_cols].notna().sum(),
    "missing": df[categorical_cols].isna().sum(),
    "unique": df[categorical_cols].nunique(dropna=True)
})

categorical_summary

C:\Users\Pc\AppData\Local\Temp\ipykernel_16160\1436619058.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(


,dtype,non_null,missing,unique
Type,str,1172,0,4
Localisation,str,1172,0,100
Other_tags,str,1172,0,21


In [91]:
categorical_cols.isnull().sum()

np.int64(0)

In [92]:
numeric_cols.isnull().sum()

np.int64(0)


### Lecture de cette étape

La compréhension initiale confirme la structure du dataset, ses types de variables et l'absence de valeurs manquantes. Les statistiques descriptives montrent toutefois une forte dispersion de `Price` et `Area`, ce qui justifie une phase de nettoyage avant toute modélisation.



<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3. Nettoyage des données</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Contrôler les doublons et préparer les variables catégorielles.</p>
</div>


In [93]:
print("Doublons exacts :", df.duplicated().sum())


Doublons exacts : 0


In [94]:
categorical_cols

Index(['Type', 'Localisation', 'Other_tags'], dtype='str')


<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.1 Nettoyage de `Other_tags`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Transformer une variable textuelle complexe en variables analytiques.</p>
</div>


### on va tranformer la colonne other tags de string a liste

In [95]:
import ast
df['Other_tags'] = df['Other_tags'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

In [96]:
tags_aplatis = df['Other_tags'].explode()
other_tags_unique = tags_aplatis.unique()
display(other_tags_unique)

other_tags_frq = tags_aplatis.value_counts()
display(other_tags_frq)

<ArrowStringArray>
[      'Nouveau', 'Moins d'un an',      'Bon état',     '10-20 ans',
      '5-10 ans',       '1-5 ans',     'À rénover',     '30-50 ans',
     '20-30 ans',             nan,     '50-70 ans']
Length: 11, dtype: str

Other_tags
Bon état         735
10-20 ans        313
Nouveau          292
5-10 ans         259
1-5 ans          131
À rénover         96
Moins d'un an     88
20-30 ans         76
30-50 ans         13
50-70 ans          2
Name: count, dtype: int64

In [97]:
current_state_tags = ['Bon état','Nouveau','À rénover']
current_state_tag_freq = other_tags_frq[current_state_tags]

age_tags = ["Moins d'un an",'1-5 ans','5-10 ans','10-20 ans','20-30 ans','30-50 ans','50-70 ans']
age_tag_freq = other_tags_frq[age_tags]

In [98]:
df = df.copy()
def extraire_categorie(liste_annonce, categorie_tags):
    for tag in liste_annonce:
        if tag in categorie_tags:
            return tag
            
    return np.nan

df['Current_state'] = df['Other_tags'].apply(lambda x: extraire_categorie(x, current_state_tags))
df['Age'] = df['Other_tags'].apply(lambda x: extraire_categorie(x, age_tags))

In [99]:
import plotly.express as px

# Graphe 1
current_state_tags = ['Bon état', 'Nouveau', 'À rénover']
state_counts = df['Current_state'].value_counts().reindex(current_state_tags).reset_index()
state_counts.columns = ['Current_state', 'count']

fig1 = px.bar(state_counts, x='Current_state', y='count', color='Current_state', 
              title="Distribution de l'État du Bien",
              labels={'Current_state': 'État', 'count': "Nombre d'annonces"},
              color_discrete_sequence=px.colors.sequential.Blues_r)
fig1.update_layout(showlegend=False)
fig1.show()

# Graphe 2
age_order = ["Moins d'un an", '1-5 ans', '5-10 ans', '10-20 ans', '20-30 ans', '30-50 ans', '50-70 ans']
age_counts = df['Age'].value_counts().reindex(age_order).reset_index()
age_counts.columns = ['Age', 'count']

fig2 = px.bar(age_counts, x='Age', y='count', color='Age', 
              title="Distribution de l'Âge du Bien",
              labels={'Age': 'Âge', 'count': "Nombre d'annonces"},
              color_discrete_sequence=px.colors.sequential.Viridis)  # Kanst3mlo Viridis hna
fig2.update_layout(showlegend=False)
fig2.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;"> Interprétation — État et âge des biens</h4>
<p style="margin:0; line-height:1.65;">Ces deux distributions permettent de comprendre la composition qualitative du parc immobilier présent dans les annonces. Elles servent aussi à vérifier si le dataset est dominé par des biens neufs, en bon état ou à rénover. Cette lecture est importante car l'état et l'ancienneté peuvent contribuer à expliquer les différences de valorisation entre biens.</p>
</div>



<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.2 Nettoyage de `Type`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Réduire les catégories rares et améliorer la robustesse de l'analyse.</p>
</div>


### Examinons maintenant la répartition des types d'annonces.

In [100]:
import plotly.express as px

type_counts = df['Type'].value_counts().reset_index()
type_counts.columns = ['Type', 'count']

fig = px.bar(type_counts, x='Type', y='count', color='Type',
             title='Distribution of Listings by Type',
             labels={'count': 'Count'})
fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;"> Interprétation — Type des biens — avant regroupement</h4>
<p style="margin:0; line-height:1.65;">Ce graphique montre la structure initiale des catégories de biens. Il permet d'identifier les catégories dominantes et les catégories peu représentées. Une catégorie très rare peut produire des estimations instables ; c'est pourquoi la phase de nettoyage regroupe ensuite certaines catégories.</p>
</div>



In [101]:
df['Type'].value_counts()

Type
Appartements    816
Villas          321
Maisons          32
Riad              3
Name: count, dtype: int64

In [102]:
df['Type'].isnull().sum()

np.int64(0)

In [103]:
df['Type'] = df['Type'].replace('Riad','Villas').replace('Maisons','Villas')

In [104]:
df['Type'].unique()

<ArrowStringArray>
['Appartements', 'Villas']
Length: 2, dtype: str

In [105]:
import plotly.express as px

print("Nouvelle distribution des types de biens :")
print(df['Type'].value_counts())

type_counts = df['Type'].value_counts().reset_index()
type_counts.columns = ['Type', 'count']

fig = px.bar(type_counts, x='Type', y='count', color='Type',
             title="Distribution des types de biens (Appartements vs Villas)",
             labels={'Type': 'Type de bien', 'count': "Nombre d'annonces"},
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(showlegend=False)
fig.show()

Nouvelle distribution des types de biens :
Type
Appartements    816
Villas          356
Name: count, dtype: int64



<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;"> Interprétation — Type des biens — après regroupement</h4>
<p style="margin:0; line-height:1.65;">Après regroupement des catégories rares, la variable `Type` devient plus homogène et plus exploitable. Cette étape facilite les comparaisons entre grandes familles de biens et réduit le risque qu'une classe très peu représentée influence excessivement l'analyse.</p>
</div>




<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.3 Nettoyage de `Localisation`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Observer la concentration des annonces par quartier.</p>
</div>


In [106]:
import plotly.express as px

top_locs = df['Localisation'].value_counts().head(27).reset_index()
top_locs.columns = ['Localisation', 'count']

fig = px.bar(top_locs, x='Localisation', y='count', color='Localisation',
             title="Distribution des annonces pour les 27 meilleurs quartiers",
             labels={'Localisation': 'Quartier (Localisation)', 'count': "Nombre d'annonces"})
fig.update_layout(showlegend=False, xaxis_tickangle=-90)
fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">Interprétation — Concentration géographique</h4>
<p style="margin:0; line-height:1.65;">La distribution des annonces par quartier permet d'identifier les zones les plus représentées dans l'échantillon. Il faut distinguer la fréquence des annonces de l'attractivité réelle d'un quartier : une forte fréquence indique surtout que cette zone est davantage représentée dans les données collectées..</p>
</div>



In [107]:
df['Localisation'].isnull().sum()

np.int64(0)


<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.4 Analyse et nettoyage de `Price`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Étudier la variable cible et ses valeurs extrêmes.</p>
</div>


### Observer les relations entre les variables et la variable cible `Price`.

In [108]:
import plotly.express as px

corr_matrix = df.corr(numeric_only=True)
print(corr_matrix)

fig = px.imshow(corr_matrix, text_auto=".2f", aspect="auto",
                color_continuous_scale='RdBu_r',
                title="Matrice de Corrélation")
fig.show()

              Price      Area     Rooms  Bedrooms  Bathrooms     Floor  \
Price      1.000000  0.907870  0.496017  0.533170   0.589295 -0.410247   
Area       0.907870  1.000000  0.501606  0.493977   0.550690 -0.406027   
Rooms      0.496017  0.501606  1.000000  0.674138   0.631745 -0.429368   
Bedrooms   0.533170  0.493977  0.674138  1.000000   0.738318 -0.409567   
Bathrooms  0.589295  0.550690  0.631745  0.738318   1.000000 -0.383340   
Floor     -0.410247 -0.406027 -0.429368 -0.409567  -0.383340  1.000000   
Price_m2   0.236767  0.001872  0.071903  0.124002   0.165653 -0.057035   

           Price_m2  
Price      0.236767  
Area       0.001872  
Rooms      0.071903  
Bedrooms   0.124002  
Bathrooms  0.165653  
Floor     -0.057035  
Price_m2   1.000000  



<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">Interprétation — Corrélations numériques</h4>
<p style="margin:0; line-height:1.65;">La matrice de corrélation sert à repérer les relations linéaires entre les variables numériques. Une corrélation élevée avec `Price` indique une association statistique, mais ne suffit pas à conclure à une causalité. Elle permet surtout d'orienter les analyses suivantes et la sélection des variables.</p>
</div>



### Analyse de la distribution de la variable cible

In [109]:
import plotly.express as px

# Graphe 1: Boxplot
fig_box = px.box(df, x='Price', title="Boxplot de Price (Avant Traitement)")
fig_box.show()

# Graphe 2: Histogramme
fig_hist = px.histogram(df, x='Price', nbins=50, title="Distribution de Price (Avant Traitement)")
fig_hist.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">Interprétation — Distribution du prix</h4>
<p style="margin:0; line-height:1.65;">Le boxplot et l'histogramme montrent la forme de la variable cible et permettent de détecter l'asymétrie ainsi que les valeurs extrêmes. Dans un marché immobilier, quelques biens très chers peuvent fortement tirer la moyenne vers le haut ; la médiane et les quantiles sont donc particulièrement utiles pour décrire le marché.</p>
</div></p>
</div>



<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">  Interprétation — Asymétrie du prix</h4>
<p style="margin:0; line-height:1.65;">La distribution du prix est fortement asymétrique vers la droite : quelques annonces très coûteuses peuvent éloigner la moyenne du comportement typique du marché. Pour les analyses descriptives, il est donc pertinent de comparer moyenne, médiane et quantiles.</p>
</div>



<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.5 Analyse et nettoyage de `Area`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Identifier les distributions extrêmes de superficie.</p>
</div>


In [110]:
import plotly.express as px
from plotly.subplots import make_subplots

fig_box = px.box(df, x='Area', color_discrete_sequence=['pink'])
fig_hist = px.histogram(df, x='Area', nbins=50, color_discrete_sequence=['pink'])

fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=('Boxplot de Area (Avant Traitement)', 'Distribution de Area (Avant Traitement)')
)

fig.add_trace(fig_box['data'][0], row=1, col=1)
fig.add_trace(fig_hist['data'][0], row=1, col=2)

fig.update_layout(
    height=500, 
    width=1000, 
    showlegend=False
)
fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;"> Interprétation — Distribution de la superficie</h4>
<p style="margin:0; line-height:1.65;">La superficie présente elle aussi des valeurs extrêmes. Le boxplot permet de repérer les annonces atypiques tandis que l'histogramme montre la concentration des surfaces. Cette variable est centrale pour l'analyse du prix et doit être contrôlée avant la modélisation.</p>
</div>



<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.6 Nettoyage de `Rooms`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Contrôler les valeurs extrêmes du nombre de pièces.</p>
</div>


In [111]:
import plotly.express as px

fig = px.histogram(
    df, 
    x='Rooms', 
    color_discrete_sequence=['skyblue'],
    title='Distribution de Rooms',
    labels={'Rooms': 'Nombre de pièces (avant traitement)'}
)

fig.update_layout(
    bargap=0.2, 
    yaxis_title="Count"
)

fig.update_xaxes(dtick=1)

fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">  Interprétation — Nombre de pièces — avant traitement</h4>
<p style="margin:0; line-height:1.65;">La distribution initiale permet de détecter les valeurs atypiques du nombre de pièces. Les valeurs très élevées sont rares et peuvent représenter des biens très particuliers ; elles sont donc contrôlées avant la suite de l'analyse.</p>
</div>


In [112]:
df['Rooms'].value_counts()

Rooms
3.0     268
4.0     259
5.0     152
2.0     108
8.0      96
6.0      96
7.0      90
9.0      38
10.0     29
1.0      20
12.0      7
11.0      4
15.0      2
14.0      1
16.0      1
13.0      1
Name: count, dtype: int64

In [113]:
df['Rooms'] = df['Rooms'].astype(int)

In [114]:
df.loc[df['Rooms'] > 10, 'Rooms'] = 10

In [115]:
print(df['Rooms'].value_counts())

Rooms
3     268
4     259
5     152
2     108
8      96
6      96
7      90
10     45
9      38
1      20
Name: count, dtype: int64


In [116]:
import plotly.express as px

fig = px.histogram(
    df, 
    x='Rooms', 
    color_discrete_sequence=['orange'],
    title='Distribution de Rooms',
    labels={'Rooms': 'Nombre de pièces (apres traitement)'}
)

fig.update_layout(
    bargap=0.2, 
    yaxis_title="Count"
)

fig.update_xaxes(dtick=1)

fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;"> Interprétation — Nombre de pièces — après traitement</h4>
<p style="margin:0; line-height:1.65;">Après traitement, la variable `Rooms` est bornée afin de limiter l'influence des valeurs extrêmes. La distribution devient plus adaptée à une analyse statistique et à un modèle prédictif.</p>
</div>



<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.7 Nettoyage de `Bedrooms`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Contrôler les valeurs extrêmes du nombre de chambres.</p>
</div>


In [117]:
import plotly.express as px

fig = px.histogram(
    df, 
    x='Bedrooms', 
    color_discrete_sequence=['pink'],
    title='Distribution de Bedrooms',
    labels={'Bedrooms': 'Nombre de pièces (avant traitement)'}
)

fig.update_layout(
    bargap=0.2, 
    yaxis_title="Count"
)

fig.update_xaxes(dtick=1)

fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;"> Interprétation — Nombre de chambres — avant traitement</h4>
<p style="margin:0; line-height:1.65;">Cette distribution permet d'identifier les modalités les plus fréquentes et les valeurs atypiques du nombre de chambres..</p>
</div>


In [118]:
df['Bedrooms'].value_counts()

Bedrooms
3.0     387
2.0     323
4.0     239
1.0     103
5.0      77
6.0      28
7.0      10
18.0      1
11.0      1
8.0       1
15.0      1
10.0      1
Name: count, dtype: int64

In [119]:
df['Bedrooms'] = df['Bedrooms'].astype(int)

In [120]:
df.loc[df['Bedrooms'] > 10, 'Bedrooms'] = 10

In [121]:
print(df['Bedrooms'].value_counts())

Bedrooms
3     387
2     323
4     239
1     103
5      77
6      28
7      10
10      4
8       1
Name: count, dtype: int64


In [122]:
import plotly.express as px

fig = px.histogram(
    df, 
    x='Bedrooms', 
    color_discrete_sequence=['skyblue'],
    title='Distribution de Bedrooms',
    labels={'Bedrooms': 'Nombre de pièces (apres traitement)'}
)

fig.update_layout(
    bargap=0.2, 
    yaxis_title="Count"
)

fig.update_xaxes(dtick=1)

fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">Interprétation — Nombre de chambres — après traitement</h4>
<p style="margin:0; line-height:1.65;">Après conversion en entier et limitation des valeurs extrêmes, `Bedrooms` est plus robuste pour les analyses comparatives et la modélisation.</p>
</div>



<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.8 Nettoyage de `Bathrooms`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Contrôler les valeurs extrêmes du nombre de salles de bain.</p>
</div>


In [123]:
import plotly.express as px

fig = px.histogram(
    df, 
    x='Bathrooms', 
    color_discrete_sequence=['skyblue'],
    title='Distribution de Bathrooms',
    labels={'Bathrooms': 'Nombre de pièces (avant traitement)'}
)

fig.update_layout(
    bargap=0.2, 
    yaxis_title="Count"
)

fig.update_xaxes(dtick=1)

fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">  Interprétation — Nombre de salles de bain — avant traitement</h4>
<p style="margin:0; line-height:1.65;">La distribution initiale permet de vérifier les modalités les plus fréquentes et de détecter les valeurs atypiques.</p>
</div>


In [124]:
df['Bathrooms'].value_counts()

Bathrooms
2.0     497
3.0     261
1.0     241
4.0     104
5.0      42
6.0      18
7.0       4
8.0       3
10.0      1
9.0       1
Name: count, dtype: int64

In [125]:
df['Bathrooms'] = df['Bathrooms'].astype(int)

In [126]:
df.loc[df['Bathrooms'] > 10 , 'Bathrooms'] = 10 

In [127]:
print(df['Bathrooms'].value_counts())

Bathrooms
2     497
3     261
1     241
4     104
5      42
6      18
7       4
8       3
10      1
9       1
Name: count, dtype: int64


In [128]:
import plotly.express as px

fig = px.histogram(
    df, 
    x='Bathrooms', 
    color_discrete_sequence=['skyblue'],
    title='Distribution de Bathrooms',
    labels={'Bathrooms': 'Nombre de pièces (apres traitement)'}
)
fig.update_layout(
    bargap=0.2, 
    yaxis_title="Count"
)
fig.update_xaxes(dtick=1)
fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;"> Interprétation — Nombre de salles de bain — après traitement</h4>
<p style="margin:0; line-height:1.65;">La variable a été convertie en entier et les valeurs extrêmes ont été limitées afin d'éviter qu'un petit nombre d'annonces atypiques domine l'analyse.</p>
</div>



<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.9 Nettoyage de `Floor`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Standardiser les étages pour limiter l'impact des valeurs atypiques.</p>
</div>


In [129]:
import plotly.express as px

fig = px.histogram(
    df, 
    x='Floor', 
    color_discrete_sequence=['skyblue'],
    title='Distribution de Floor',
    labels={'Floor': 'Nombre de pièces (avant traitement)'}
)
fig.update_layout(
    bargap=0.2, 
    yaxis_title="Count"
)
fig.update_xaxes(dtick=1)
fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">Interprétation — Distribution des étages — avant traitement</h4>
<p style="margin:0; line-height:1.65;">La distribution initiale montre comment les annonces se répartissent selon l'étage. Elle permet aussi de repérer des valeurs inhabituelles ou très peu représentées.</p>
</div>


In [130]:
df['Floor'].value_counts()

Floor
0.0     356
3.0     182
1.0     155
4.0     154
2.0     147
5.0     111
6.0      28
7.0      21
8.0       6
10.0      4
12.0      3
9.0       2
11.0      1
20.0      1
15.0      1
Name: count, dtype: int64

In [131]:
df['Floor'] = df['Floor'].astype(int)

In [132]:
df.loc[df['Floor'] > 4 , 'Floor'] = 4

In [133]:
print(df['Floor'].value_counts())

Floor
0    356
4    332
3    182
1    155
2    147
Name: count, dtype: int64


In [134]:
import plotly.express as px

fig = px.histogram(
    df, 
    x='Floor', 
    color_discrete_sequence=['skyblue'],
    title='Distribution de Floor',
    labels={'Floor': 'Nombre de pièces (apres traitement)'}
)
fig.update_layout(
    bargap=0.2, 
    yaxis_title="Count"
)
fig.update_xaxes(dtick=1)
fig.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;"> Interprétation — Distribution des étages — après traitement</h4>
<p style="margin:0; line-height:1.65;">Après limitation des étages élevés, la variable `Floor` est plus stable et plus simple à exploiter dans les modèles.</p>
</div>



<div style="background:#fff7ed; padding:22px 26px; border-radius:14px; border-left:6px solid #d97706; margin:18px 0;">
<h1 style="color:#d97706; margin:0 0 4px 0;">3.10 Validation de `Price_m2`</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Vérifier la cohérence entre prix, superficie et prix au m².</p>
</div>


### Vérification de cohérence entre `Price`, `Area` et `Price_m2`.

### verefication de coherence 

In [135]:
prix_calcule = round(df['Price'] / df['Area'], 2)
prix_existant = round(df['Price_m2'], 2)

lignes_incoherentes = df[prix_calcule != prix_existant]

print(f"combien des lignes ont des erreurs : {len(lignes_incoherentes)}")

if len(lignes_incoherentes) > 0:
    print(lignes_incoherentes[['Price', 'Area', 'Price_m2']].head())

combien des lignes ont des erreurs : 0


In [136]:
fig_box = px.box(df, x='Price_m2', title="Boxplot de Price_m2 ")
fig_box.show()


<div style="background:#408bed; padding:18px 22px; border-radius:12px; border:1px solid #f0f3f7; margin:14px 0;">
<h4 style="color:#f0f3f7; margin:0 0 8px 0;">  Interprétation — Prix au m²</h4>
<p style="margin:0; line-height:1.65;">Le boxplot de `Price_m2` permet de contrôler la dispersion de la valeur la plus directement comparable entre biens de superficies différentes. Les valeurs extrêmes doivent être interprétées avec prudence : elles peuvent correspondre à des biens réellement premium, mais aussi à des annonces atypiques</p>
</div>



### Bilan du Data Cleaning

Les opérations réalisées ont permis de passer d'une structure brute à des variables plus directement exploitables :

- contrôle des doublons et des valeurs manquantes ;
- transformation de `Other_tags` en variables `Current_state` et `Age` ;
- regroupement des catégories rares de `Type` ;
- contrôle des distributions de `Price`, `Area`, `Rooms`, `Bedrooms`, `Bathrooms` et `Floor` ;
- vérification de la cohérence de `Price_m2`.

**Point de vigilance :** le nettoyage des valeurs extrêmes doit toujours être interprété comme une décision analytique. Une valeur atypique n'est pas automatiquement une erreur ; elle peut représenter un bien réellement exceptionnel.



<div style="background:#ecfdf5; padding:22px 26px; border-radius:14px; border-left:6px solid #0f766e; margin:18px 0;">
<h1 style="color:#0f766e; margin:0 0 4px 0;">4. Données finales prêtes pour la modélisation</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Exporter et préparer les features.</p>
</div>


In [137]:
df.to_csv('data_cleaned.csv', index=False)


<div style="background:#ecfeff; padding:22px 26px; border-radius:14px; border-left:6px solid #0369a1; margin:18px 0;">
<h1 style="color:#0369a1; margin:0 0 4px 0;">5. Questions Data Analyst — faire parler les données</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Passer de la visualisation à des réponses concrètes</p>
</div>



## 🎯 Objectif de cette partie

Un Data Analyst ne se limite pas à produire des graphiques. Il doit être capable de transformer une question métier en indicateur, analyse ou comparaison mesurable.

Les questions ci-dessous sont volontairement construites **à partir des variables réellement présentes dans le dataset**. Les réponses sont calculées automatiquement à partir de `df`, afin d'éviter d'introduire des chiffres fictifs dans le rapport.



### Question 1 — Quel est le profil général du marché observé ?

**Objectif :** mesurer le volume d'annonces, le nombre de quartiers et la répartition des types de biens.

**Indicateurs :**
- nombre total d'annonces ;
- nombre de localisations ;
- nombre de types ;
- type dominant.


In [138]:

n_annonces = len(df)
n_quartiers = df['Localisation'].nunique()
n_types = df['Type'].nunique()
type_dominant = df['Type'].value_counts().idxmax()
part_type_dominant = df['Type'].value_counts(normalize=True).max() * 100

print(f"Nombre d'annonces : {n_annonces:,}")
print(f"Nombre de localisations : {n_quartiers:,}")
print(f"Nombre de types de biens : {n_types:,}")
print(f"Type dominant : {type_dominant} ({part_type_dominant:.1f}% des annonces)")


Nombre d'annonces : 1,172
Nombre de localisations : 100
Nombre de types de biens : 2
Type dominant : Appartements (69.6% des annonces)



### Question 2 — Quels quartiers affichent les prix au m² les plus élevés ?

**Objectif :** comparer les quartiers sur une base homogène. Le prix total dépend fortement de la superficie ; `Price_m2` est donc plus adapté pour comparer le niveau de valorisation.

Pour éviter qu'un quartier avec très peu d'annonces soit classé uniquement à cause d'une observation atypique, on utilise ici un **minimum de 5 annonces** et la **médiane**.


In [139]:

quartiers_prix = (
    df.groupby('Localisation')
      .agg(
          Nb_annonces=('Price_m2', 'size'),
          Prix_m2_median=('Price_m2', 'median'),
          Prix_m2_moyen=('Price_m2', 'mean')
      )
      .query('Nb_annonces >= 5')
      .sort_values('Prix_m2_median', ascending=False)
)

display(quartiers_prix.head(10))


,Nb_annonces,Prix_m2_median,Prix_m2_moyen
Localisation,,,
Casablanca Finance City,12,22853.550123,27140.955239
Anfa Supérieur,17,21604.938272,22057.065182
Hermitage,13,21163.043478,18600.631938
Racine Extension,11,19067.796610,19295.706452
Racine,62,18901.362931,20571.620134
Ain Diab Extension,15,18481.012658,17541.251096
Triangle d'Or,21,18181.818182,18182.610617
Riviera,5,18125.000000,17266.666667
Anfa,43,18000.000000,18195.849424



### Question 3 — La superficie est-elle associée au prix de vente ?

**Objectif :** mesurer la relation entre `Area` et `Price`.

> Une corrélation positive indique que les biens plus grands tendent à être plus chers dans l'échantillon. Elle ne signifie pas que la superficie est l'unique facteur déterminant du prix.


In [140]:

corr_area_price = df['Area'].corr(df['Price'])
print(f"Corrélation Area / Price : {corr_area_price:.3f}")

if corr_area_price >= 0.7:
    interpretation = "relation linéaire positive forte"
elif corr_area_price >= 0.4:
    interpretation = "relation linéaire positive modérée"
elif corr_area_price > 0:
    interpretation = "relation linéaire positive faible"
else:
    interpretation = "absence de relation linéaire positive"

print(f"Lecture : {interpretation}.")


Corrélation Area / Price : 0.908
Lecture : relation linéaire positive forte.



### Question 4 — Quel type de bien présente la valeur médiane la plus élevée ?

**Objectif :** comparer les catégories sur trois dimensions : prix total, superficie et prix au m².


In [141]:

comparaison_type = (
    df.groupby('Type')
      .agg(
          Nb_annonces=('Price', 'size'),
          Prix_median=('Price', 'median'),
          Surface_mediane=('Area', 'median'),
          Prix_m2_median=('Price_m2', 'median')
      )
      .sort_values('Prix_median', ascending=False)
)

display(comparaison_type)


,Nb_annonces,Prix_median,Surface_mediane,Prix_m2_median
Type,,,,
Villas,356,8750000.0,561.5,16525.862069
Appartements,816,1600000.0,108.5,14656.565657



### Question 5 — L'état du bien semble-t-il associé à sa valorisation ?

**Objectif :** comparer `Current_state` selon le prix total et le prix au m².

Cette analyse décrit une **association observée** dans les annonces ; elle ne permet pas, à elle seule, de conclure que l'état cause directement la différence de prix.


In [142]:

etat_analyse = (
    df.groupby('Current_state')
      .agg(
          Nb_annonces=('Price', 'size'),
          Prix_median=('Price', 'median'),
          Prix_m2_median=('Price_m2', 'median')
      )
      .sort_values('Prix_m2_median', ascending=False)
)

display(etat_analyse)


,Nb_annonces,Prix_median,Prix_m2_median
Current_state,,,
Nouveau,292,1830000.0,16666.666667
À rénover,96,8600000.0,15131.578947
Bon état,735,2200000.0,14410.480349



### Question 6 — Quels biens représentent le segment très haut de gamme ?

**Objectif :** définir le segment premium directement à partir de la distribution observée, plutôt que d'imposer un seuil arbitraire.

Ici, le segment premium correspond aux **10 % des annonces ayant les prix les plus élevés**.


In [143]:

seuil_premium = df['Price'].quantile(0.90)
premium = df[df['Price'] >= seuil_premium].copy()

print(f"Seuil du top 10% : {seuil_premium:,.0f}")
print(f"Nombre d'annonces premium : {len(premium):,}")

resume_premium = {
    "Prix médian": premium['Price'].median(),
    "Surface médiane": premium['Area'].median(),
    "Prix/m² médian": premium['Price_m2'].median(),
    "Pièces médianes": premium['Rooms'].median()
}
display(pd.Series(resume_premium).to_frame("Valeur"))


Seuil du top 10% : 12,000,000
Nombre d'annonces premium : 126


,Valeur
Prix médian,1.625000e+07
Surface médiane,1.058000e+03
Prix/m² médian,1.652586e+04
Pièces médianes,8.000000e+00



### Question 7 — Quelles caractéristiques distinguent les annonces les plus chères ?

**Objectif :** comparer le segment premium au reste du marché sur les variables numériques disponibles.

Cette étape permet de passer d'une simple description à une **lecture comparative** utile pour un décideur immobilier.


In [144]:

seuil_premium = df['Price'].quantile(0.90)
df_analysis = df.copy()
df_analysis['Segment'] = np.where(df_analysis['Price'] >= seuil_premium, 'Premium (Top 10%)', 'Reste du marché')

comparatif_segments = (
    df_analysis.groupby('Segment')[['Price', 'Area', 'Rooms', 'Bedrooms', 'Bathrooms', 'Price_m2']]
               .median()
               .T
)

display(comparatif_segments)


Segment,Premium (Top 10%),Reste du marché
Price,1.625000e+07,1950000.0
Area,1.058000e+03,126.5
Rooms,8.000000e+00,4.0
Bedrooms,4.000000e+00,3.0
Bathrooms,4.000000e+00,2.0
Price_m2,1.652586e+04,15000.0



###  Synthèse Data Analyst

À la fin de cette section, l'analyse doit permettre de répondre à quatre axes de décision :

1. **Où ?** → quels quartiers sont les plus représentés et lesquels affichent les niveaux de prix au m² les plus élevés ;
2. **Quoi ?** → quels types de biens dominent le marché et lesquels sont les plus valorisés ;
3. **Pourquoi / quels facteurs ?** → quelles caractéristiques sont associées au prix, notamment la superficie et les caractéristiques du bien ;
4. **Quel segment ?** → comment caractériser les biens du segment premium par rapport au reste du marché.

Cette étape constitue le pont entre **EDA (Exploratory Data Analysis)** et **Machine Learning** : les observations servent ensuite à sélectionner, transformer et interpréter les variables utilisées dans les modèles.



<div style="background:#f5f3ff; padding:22px 26px; border-radius:14px; border-left:6px solid #7c3aed; margin:18px 0;">
<h1 style="color:#7c3aed; margin:0 0 4px 0;">5. Préparation du Machine Learning</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Encodage, séparation train/test et mise à l'échelle.</p>
</div>


In [145]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1172 entries, 0 to 1171
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Type           1172 non-null   str    
 1   Localisation   1172 non-null   str    
 2   Price          1172 non-null   int64  
 3   Area           1172 non-null   float64
 4   Rooms          1172 non-null   int64  
 5   Bedrooms       1172 non-null   int64  
 6   Bathrooms      1172 non-null   int64  
 7   Floor          1172 non-null   int64  
 8   Other_tags     1172 non-null   object 
 9   Price_m2       1172 non-null   float64
 10  Current_state  1123 non-null   str    
 11  Age            882 non-null    str    
dtypes: float64(2), int64(5), object(1), str(4)
memory usage: 151.4+ KB


In [146]:
cols_to_keep = ['Type','Localisation','Area','Rooms','Bedrooms','Bathrooms','Floor','Current_state','Age','Price']
df = df[cols_to_keep]

In [147]:
display(df.isna().sum())

Type               0
Localisation       0
Area               0
Rooms              0
Bedrooms           0
Bathrooms          0
Floor              0
Current_state     49
Age              290
Price              0
dtype: int64

In [148]:
features = ['Type','Localisation','Area','Rooms','Bedrooms','Bathrooms','Floor','Current_state','Age']
X = df[features]
y = df['Price'].values 

In [149]:
cols_to_impute = ['Current_state', 'Age']

df = df.copy()
encoder = OrdinalEncoder()
df[cols_to_impute] = encoder.fit_transform(df[cols_to_impute])

numeric_cols = ['Area', 'Rooms', 'Bedrooms', 'Bathrooms', 'Floor', 'Current_state', 'Age']
df = df[numeric_cols]

imputer = KNNImputer(n_neighbors=5, weights='distance')
df_knn_imputed = pd.DataFrame(imputer.fit_transform(df), columns=numeric_cols)

df['Current_state'] = df_knn_imputed['Current_state']
df['Age'] = df_knn_imputed['Age']

df[cols_to_impute] = encoder.inverse_transform(df[cols_to_impute])

print("Les valeurs manquantes b3d KNNImputer:")
display(df.isna().sum())

Les valeurs manquantes b3d KNNImputer:


Area             0
Rooms            0
Bedrooms         0
Bathrooms        0
Floor            0
Current_state    0
Age              0
dtype: int64

In [150]:
x = pd.get_dummies(X)
x

,Area,Rooms,Bedrooms,Bathrooms,Floor,Type_Appartements,Type_Villas,Localisation_Ain Borja,Localisation_Ain Chock,Localisation_Ain Diab,...,Current_state_Bon état,Current_state_Nouveau,Current_state_À rénover,Age_1-5 ans,Age_10-20 ans,Age_20-30 ans,Age_30-50 ans,Age_5-10 ans,Age_50-70 ans,Age_Moins d'un an
0,219.0,4,3,3,4,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,True
1,86.0,3,2,2,1,True,False,False,False,False,...,True,False,False,False,True,False,False,False,False,False
2,86.0,3,2,1,4,True,False,False,False,False,...,True,False,False,False,True,False,False,False,False,False
3,180.0,5,3,3,3,True,False,False,False,False,...,True,False,False,False,True,False,False,False,False,False
4,120.0,5,3,2,4,True,False,False,False,False,...,True,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167,1040.0,8,5,6,0,False,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False
1168,74.0,3,2,2,4,True,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
1169,56.0,2,1,2,4,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,True
1170,68.0,3,2,1,4,True,False,False,False,False,...,True,False,False,False,False,False,False,True,False,False


In [151]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size = 0.3,random_state = 123)

In [152]:
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),columns = X_train.columns)
X_test_scaled = pd.DataFrame(scaler.fit_transform(X_test),columns = X_test.columns)

In [153]:
df.head()

,Area,Rooms,Bedrooms,Bathrooms,Floor,Current_state,Age
0,219.0,4,3,3,4,Nouveau,Moins d'un an
1,86.0,3,2,2,1,Bon état,10-20 ans
2,86.0,3,2,1,4,Bon état,10-20 ans
3,180.0,5,3,3,3,Bon état,10-20 ans
4,120.0,5,3,2,4,Bon état,5-10 ans



<div style="background:#fff1f2; padding:22px 26px; border-radius:14px; border-left:6px solid #be123c; margin:18px 0;">
<h1 style="color:#be123c; margin:0 0 4px 0;">6. Optimisation des modèles</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Comparer les modèles et limiter le risque de surapprentissage.</p>
</div>


In [154]:
def optimize_models_anti_overfitting(X_train, X_test, y_train, y_test):
    models_grids = {
        "Ridge (Scaled)": {
            "model": Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())]),
            "params": {"ridge__alpha": [1.0, 10.0, 50.0, 100.0]}
        },
        "Lasso (Scaled)": {
            "model": Pipeline([("scaler", StandardScaler()), ("lasso", Lasso())]),
            "params": {"lasso__alpha": [0.001, 0.01, 0.1, 1.0]}
        },
        "KNN (Scaled)": {
            "model": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())]),
            "params": {
                "knn__n_neighbors": [5, 9, 15], 
                "knn__weights": ['uniform'] 
            }
        },
        "SVR (Scaled)": {
            "model": Pipeline([("scaler", StandardScaler()), ("svr", SVR())]),
            "params": {
                "svr__kernel": ['rbf'], 
                "svr__C": [0.1, 1.0, 10.0], 
                "svr__epsilon": [0.01, 0.1]
            }
        },
        "Decision Tree": {
            "model": DecisionTreeRegressor(random_state=42),
            "params": {
                "max_depth": [4, 6, 8], 
                "min_samples_split": [10, 20], 
                "min_samples_leaf": [5, 10]
            }
        },
        "Random Forest": {
            "model": RandomForestRegressor(random_state=42, n_jobs=-1),
            "params": {
                "n_estimators": [100, 200], 
                "max_depth": [5, 8, 10], 
                "min_samples_split": [10, 15], 
                "min_samples_leaf": [5, 10]
            }
        },
        "Gradient Boosting": {
            "model": GradientBoostingRegressor(random_state=42),
            "params": {
                "n_estimators": [100, 200], 
                "learning_rate": [0.01, 0.05], 
                "max_depth": [3, 5]
            }
        },
        "XGBoost": {
            "model": XGBRegressor(random_state=42, n_jobs=-1),
            "params": {
                "n_estimators": [100, 200], 
                "learning_rate": [0.01, 0.05], 
                "max_depth": [3, 5], 
                "subsample": [0.7, 0.8], 
                "colsample_bytree": [0.7, 0.8]
            }
        },
        "CatBoost": {
            "model": CatBoostRegressor(verbose=0, random_state=42),
            "params": {
                "iterations": [200, 500], 
                "learning_rate": [0.01, 0.05], 
                "depth": [4, 6], 
                "l2_leaf_reg": [5, 10, 20] 
            }
        }
    }
    
    results = []
    best_models_dict = {}
    
    for name, config in models_grids.items():
        print(f"⏳ Optimisation en cours pour : {name}...")
        
        grid_search = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            scoring='r2',
            cv=5,
            n_jobs=-1
        )
        
        grid_search.fit(X_train, y_train)
        
        best_model = grid_search.best_estimator_
        best_models_dict[name] = best_model
        
        y_train_pred = best_model.predict(X_train)
        y_test_pred = best_model.predict(X_test)
        
        results.append({
            "Modèle": name,
            "R2 Score (Train)": round(r2_score(y_train, y_train_pred), 4),
            "R2 Score (Test)": round(r2_score(y_test, y_test_pred), 4),
            "RMSE": round(math.sqrt(mean_squared_error(y_test, y_test_pred)), 4),
            "MAE": round(mean_absolute_error(y_test, y_test_pred), 4),
            "Meilleurs Paramètres": str(grid_search.best_params_)
        })
        
    df_results = pd.DataFrame(results).sort_values(by="R2 Score (Test)", ascending=False).reset_index(drop=True)
    
    print("\n✅ L'optimisation salat ! Hada l'classement jdid:")
    return df_results, best_models_dict


In [155]:
tableau_final, meilleurs_modeles = optimize_models_anti_overfitting(X_train, X_test, y_train, y_test)

display(tableau_final)



⏳ Optimisation en cours pour : Ridge (Scaled)...
⏳ Optimisation en cours pour : Lasso (Scaled)...


c:\Users\Pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.289467e+14, tolerance: 4.275e+12
  model = cd_fast.enet_coordinate_descent(


⏳ Optimisation en cours pour : KNN (Scaled)...
⏳ Optimisation en cours pour : SVR (Scaled)...
⏳ Optimisation en cours pour : Decision Tree...
⏳ Optimisation en cours pour : Random Forest...
⏳ Optimisation en cours pour : Gradient Boosting...
⏳ Optimisation en cours pour : XGBoost...
⏳ Optimisation en cours pour : CatBoost...

✅ L'optimisation salat ! Hada l'classement jdid:


,Modèle,R2 Score (Train),R2 Score (Test),RMSE,MAE,Meilleurs Paramètres
0,CatBoost,0.9747,0.8032,3.457116e+06,1.116853e+06,"{'depth': 4, 'iterations': 500, 'l2_leaf_reg':..."
1,Gradient Boosting,0.9724,0.7974,3.508220e+06,1.169833e+06,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est..."
2,Ridge (Scaled),0.9275,0.7824,3.635513e+06,1.229744e+06,{'ridge__alpha': 10.0}
3,Lasso (Scaled),0.9277,0.7818,3.640412e+06,1.231791e+06,{'lasso__alpha': 1.0}
4,XGBoost,0.9640,0.7750,3.696849e+06,1.224730e+06,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."
5,Decision Tree,0.8947,0.7363,4.001861e+06,1.336517e+06,"{'max_depth': 8, 'min_samples_leaf': 10, 'min_..."
6,Random Forest,0.9059,0.7281,4.064215e+06,1.309031e+06,"{'max_depth': 5, 'min_samples_leaf': 5, 'min_s..."
7,KNN (Scaled),0.7644,0.5794,5.054163e+06,2.145554e+06,"{'knn__n_neighbors': 9, 'knn__weights': 'unifo..."
8,SVR (Scaled),-0.1220,-0.1288,8.280492e+06,3.924402e+06,"{'svr__C': 10.0, 'svr__epsilon': 0.1, 'svr__ke..."



<div style="background:#fff1f2; padding:22px 26px; border-radius:14px; border-left:6px solid #be123c; margin:18px 0;">
<h1 style="color:#be123c; margin:0 0 4px 0;">sauvgardement de meilleur model AI </h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>Comparer les modèles et limiter le risque de surapprentissage.</p>
</div>

In [156]:
best_catboost = meilleurs_modeles["CatBoost"]

In [157]:
# joblib.dump(best_catboost, 'modele_MaisonDeLUX.pkl')


<div style="background:#eff6ff; padding:22px 26px; border-radius:14px; border-left:6px solid #1d4ed8; margin:18px 0;">
<h1 style="color:#1d4ed8; margin:0 0 4px 0;">7. Conclusion générale</h1>
<p style='margin:6px 0 0 0;color:#475569;font-size:1.0rem'>De la donnée brute à une base prête pour l'estimation</p>
</div>



### Conclusion

Ce projet suit une démarche complète d'analyse de données immobilières :

**Données brutes → Compréhension → Nettoyage → Analyse exploratoire → Questions métier → Préparation ML → Modélisation**

L'analyse descriptive permet d'abord de comprendre la structure du marché observé et les principales sources de dispersion des prix. Le nettoyage transforme ensuite les informations brutes en variables plus pertinentes, notamment à partir de `Other_tags`.

La section **Data Analyst** apporte une dimension décisionnelle au projet : au lieu de seulement montrer des distributions, elle formule des questions et produit des indicateurs permettant de comparer les quartiers, les types de biens, la superficie, l'état des biens et le segment premium.

Enfin, la préparation Machine Learning utilise cette base nettoyée pour comparer plusieurs modèles d'estimation du prix.

> **Limite importante :** les conclusions décrivent l'échantillon d'annonces collecté sur Mubawab ; elles ne doivent pas être interprétées automatiquement comme une mesure exhaustive de l'ensemble du marché immobilier de Casablanca.
